In [5]:
import pandas as pd
df = pd.read_csv('./results/hmm_regimes_ALL_pairs.csv')
df['date'] = pd.to_datetime(df['date'])
print("SHAPE:", df.shape)
print("COLUMNS:", list(df.columns))
print("\nFIRST 3 ROWS:\n", df.head(3))
print("\nUNIQUE PAIRS:", len(df['pair'].unique()))


SHAPE: (61643, 6)
COLUMNS: ['pair', 'date', 'regime', 'max_prob', 'spread', 'spread_z']

FIRST 3 ROWS:
        pair       date  regime  max_prob    spread  spread_z
0  ABNB_APD 2021-06-04       0  1.000000  0.015783  0.287926
1  ABNB_APD 2021-06-07       0  0.993741  0.007996  0.153388
2  ABNB_APD 2021-06-08       0  0.959658 -0.001606 -0.012509

UNIQUE PAIRS: 19


In [6]:
import numpy as np


df['z_r'] = df['spread_z'] 


df['signal'] = 0
df.loc[(df['z_r'] < -2.5) & (df['max_prob'] > 0.8), 'signal'] = -1  # Long spread
df.loc[(df['z_r'] > 2.5) & (df['max_prob'] > 0.8), 'signal'] = 1   # Short spread


df['position'] = df.groupby('pair')['signal'].transform(
    lambda x: x.replace(0, np.nan).ffill().fillna(0)
)


df['size'] = np.clip(df['position'] * 0.05, -0.05, 0.05)


df.to_csv('./results/signals_ALL_pairs.csv', index=False)
print("Fixrd signals saved")
print("Long:", len(df[df['signal']==1]), "Short:", len(df[df['signal']==-1]))
print("Flips avg:", df.groupby('pair')['position'].apply(lambda x: (x.diff() != 0).sum()).mean())


Fixrd signals saved
Long: 652 Short: 856
Flips avg: 16.210526315789473


In [7]:
import plotly.graph_objects as go

for pair in df['pair'].unique():
    pair_df = df[df['pair'] == pair].sort_values('date')
    
    fig = go.Figure()
    
  
    colors = {0:'blue', 1:'orange', 2:'green', 3:'red'}
    for regime in sorted(pair_df['regime'].unique()):
        regime_data = pair_df[pair_df['regime'] == regime]
        fig.add_trace(go.Scatter(x=regime_data['date'], y=regime_data['z_r'], 
                                mode='lines', name=f'Regime {int(regime)}',
                                line=dict(color=colors[regime], width=1.8)))
   
    fig.add_hline(y=2,  line_dash="dash", line_color="black", annotation_text="ENTRY")
    fig.add_hline(y=-2, line_dash="dash", line_color="black")
    fig.add_hline(y=0,  line_dash="dot",  line_color="grey")
    
  
    longs  = pair_df[pair_df['signal'] == 1]
    shorts = pair_df[pair_df['signal'] == -1]
    if len(longs) > 0:
        fig.add_trace(go.Scatter(x=longs['date'], y=longs['z_r'], 
                                mode='markers', marker=dict(color='green', size=12, symbol='triangle-up'),
                                name=f'LONG n={len(longs)}'))
    if len(shorts) > 0:
        fig.add_trace(go.Scatter(x=shorts['date'], y=shorts['z_r'], 
                                mode='markers', marker=dict(color='red', size=12, symbol='triangle-down'),
                                name=f'SHORT n={len(shorts)}'))
    
    fig.update_layout(
        title=f'{pair} - Regime z_r + Trading Signals',
        xaxis_title='Date', yaxis_title='z_r = (spread - μ_r)/σ_r',
        height=500, width=1400, showlegend=True, plot_bgcolor='white'
    )
    
    fig.write_image(f"./results/{pair}_signals.png", width=1400, height=500, scale=2)
    print(f"{pair} saved")
    fig.show()


ABNB_APD saved


APP_COF saved


CRM_CI saved


HON_SHW saved


HON_SYK saved


...

SCHW_HLT saved


SYK_INTU saved


TMO_HON saved


TXN_SPGI saved


UBER_ECL saved


In [8]:
summary = df.groupby('pair').agg({
    'signal': lambda x: (x==1).sum(),
    'signal': lambda x: (x==-1).sum(), 
    'regime': lambda x: x.nunique(),
    'z_r': ['mean', 'std'],
    'max_prob': 'mean'
}).round(3)

print("signal summary per pair:")
print(summary)
summary.to_csv('./results/signal_summary.csv')
print("\nSummary saved - ready for backtest")


signal summary per pair:
           signal   regime  z_r      max_prob
         <lambda> <lambda> mean  std     mean
pair                                         
ABNB_APD       22        4 -0.0  1.0    0.921
APP_COF         3        4 -0.0  1.0    0.945
CRM_CI         46        4 -0.0  1.0    0.922
HON_SHW        60        4 -0.0  1.0    0.879
HON_SYK        43        4 -0.0  1.0    0.906
JPM_ANET       35        4 -0.0  1.0    0.912
LIN_QCOM       47        4  0.0  1.0    0.856
MA_AON         64        4  0.0  1.0    0.905
MA_NSC         64        4 -0.0  1.0    0.899
MA_UBER        27        4 -0.0  1.0    0.903
MCK_DELL       36        4  0.0  1.0    0.911
MRK_UNP        64        4 -0.0  1.0    0.864
NEE_TMO        51        4 -0.0  1.0    0.883
ORCL_PWR       56        4 -0.0  1.0    0.921
SCHW_HLT       53        4 -0.0  1.0    0.877
SYK_INTU       65        4 -0.0  1.0    0.900
TMO_HON        66        4 -0.0  1.0    0.889
TXN_SPGI       41        4  0.0  1.0    0.896
UBER_ECL 